In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

In [3]:
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Exploring games data

In [4]:
clean_games_fp = Path(config['games_folder'] + config['games_clean_fp'])

games_df = pd.read_json(clean_games_fp, orient='records')

games_df.head()

,name,rating,updated_at,first_release_date,id,game_modes_Battle Royale,game_modes_Co-operative,game_modes_Massively Multiplayer Online (MMO),game_modes_Multiplayer,game_modes_Single player,...,platforms_WonderSwan Color,platforms_Xbox,platforms_Xbox 360,platforms_Xbox One,platforms_Xbox Series X|S,platforms_ZX Spectrum,platforms_Zeebo,platforms_e-Reader / Card-e Reader,platforms_iOS,platforms_visionOS
0,Tempest Rising,80.089799,2026-04-24 01:59:53,1970-01-01T00:00:01.744,213240,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,DuneCrawl,NaN,2026-04-24 01:59:43,1970-01-01T00:00:01.767,318505,0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
2,Dead by Daylight: Endless Hunt Pack,NaN,2026-04-24 01:58:17,1970-01-01T00:00:01.715,301204,0,1,0,1,1,...,0,0,0,1,1,0,0,0,0,0
3,Hot Wheels Unleashed 2: Turbocharged,65.235925,2026-04-24 01:55:01,1970-01-01T00:00:01.697,251471,0,0,0,1,1,...,0,0,0,1,1,0,0,0,0,0
4,Chop Chop Inc.,NaN,2026-04-24 01:55:00,1970-01-01T00:00:01.798,398967,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [5]:
#Checking to make sure the id uniquely identifies each row
print("No duplicate IDS") if len(games_df['id'].unique()) == len(games_df) else print("Duplicate IDs found")

No duplicate IDS


In [6]:
#Checking to see if duplicate game names exist
print("No duplicate game names") if len(games_df['name'].unique()) == len(games_df) else print("Duplicate game names found")

Duplicate game names found


In [7]:
#Get all rows with duplicate game names
duplicate_names_df = games_df[games_df.duplicated(subset=['name'], keep=False)].sort_values('name')
duplicate_names_df.head(6)

,name,rating,updated_at,first_release_date,id,game_modes_Battle Royale,game_modes_Co-operative,game_modes_Massively Multiplayer Online (MMO),game_modes_Multiplayer,game_modes_Single player,...,platforms_WonderSwan Color,platforms_Xbox,platforms_Xbox 360,platforms_Xbox One,platforms_Xbox Series X|S,platforms_ZX Spectrum,platforms_Zeebo,platforms_e-Reader / Card-e Reader,platforms_iOS,platforms_visionOS
16940,10-Pin Bowling,NaN,2024-11-14 09:27:27,1970-01-01T00:00:00.933,92273,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
16939,10-Pin Bowling,NaN,2024-11-14 09:33:09,1970-01-01T00:00:00.473,153453,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
12752,15 Minutes,NaN,2026-02-02 16:52:35,1970-01-01T00:00:01.761,355071,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
6353,15 Minutes,NaN,2026-04-09 19:38:22,1970-01-01T00:00:01.767,395433,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
5539,1942,67.742592,2026-04-13 08:03:19,1970-01-01T00:00:00.470,6075,0,1,0,0,1,...,0,0,0,0,0,1,0,0,0,0
4280,1942,61.392350,2026-04-18 08:37:59,1970-01-01T00:00:00.503,272544,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0


In [8]:
#Find columns where duplicates differ (excluding name, rating, updated_at, id)
exclude_cols = {'name', 'rating', 'updated_at', 'id'}
cols_to_check = [col for col in duplicate_names_df.columns if col not in exclude_cols]

i = 0
for game_name in duplicate_names_df['name'].unique():
    game_group = duplicate_names_df[duplicate_names_df['name'] == game_name]
    differing_cols = []
    for col in cols_to_check:
        if len(game_group[col].unique()) > 1:
            differing_cols.append(col)
    if'first_release_date' not in differing_cols:  # Print every 10th game to avoid too much output
        #print(f"{game_name}: {differing_cols}")
        pass
    i+=1

There are many duplicates but they may functionally differ so we'll keep them for now

# Exploring multiplayer modes data

In [9]:
clean_modes_fp = Path(config['multiplayer_modes_folder'] + config['multiplayer_modes_clean_fp'])

modes_df = pd.read_json(clean_modes_fp, orient='records')

print("Contains " + str(len(modes_df)) + " rows")
modes_df.head()

Contains 24028 rows


,id,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,platform,splitscreen
0,9953,92273,False,False,False,-1,2,False,-1,-1,Game Boy Color,False
1,1832,7153,True,True,True,2,0,False,0,0,Xbox 360,False
2,7987,57887,False,False,True,0,2,False,0,0,Xbox One,False
3,7,46076,False,False,False,-1,30,False,-1,-1,PC (Microsoft Windows),False
4,10207,31256,False,False,False,-1,-1,False,-1,-1,Web browser,False


In [10]:
def isunique(df, subset):
    return len(df[subset].unique()) == len(df)

print("No duplicate IDS") if isunique(modes_df, 'id') else print("Duplicate IDs found")
print("No duplicate game ids") if isunique(modes_df, 'game') else print("Duplicate game ids found")

No duplicate IDS
Duplicate game ids found


In [12]:
duplicate_game_ids_df = modes_df[modes_df.duplicated(subset=['game'], keep=False)].sort_values('game')
duplicate_game_ids_df = duplicate_game_ids_df.merge(games_df[['id', 'name']], left_on='game', right_on='id', how='left', suffixes=('_mode', '_game'))
duplicate_game_ids_df.head(10)

,id_mode,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,platform,splitscreen,id_game,name
0,11594,72,False,False,True,2,0,True,2,0,Mac,True,72.0,Portal 2
1,11591,72,False,False,True,2,0,True,2,0,Xbox 360,True,72.0,Portal 2
2,11592,72,False,False,True,2,0,True,2,0,PlayStation 3,True,72.0,Portal 2
3,11593,72,False,False,True,2,0,True,2,0,PC (Microsoft Windows),True,72.0,Portal 2
4,11595,72,False,False,True,2,0,True,2,0,Linux,True,72.0,Portal 2
5,17435,83,False,False,False,2,-1,False,-1,-1,PC (Microsoft Windows),True,83.0,Baldur's Gate: Dark Alliance
6,1631,83,False,True,True,2,0,False,0,0,PlayStation 2,False,83.0,Baldur's Gate: Dark Alliance
7,11137,121,True,True,False,0,0,False,0,0,PC (Microsoft Windows),False,121.0,Minecraft: Java Edition
8,11139,121,True,True,False,0,0,False,0,0,Linux,False,121.0,Minecraft: Java Edition
9,11138,121,True,True,False,0,0,False,0,0,Mac,False,121.0,Minecraft: Java Edition


Duplicate IDS often seems to indicate multi-platform releases

In [ ]:
full_df = modes_df.merge(games_df, left_on='game', right_on='id', how='left', suffixes=('_mode', '_game'))
full_df.head()

,id_mode,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,...,platforms_WonderSwan Color,platforms_Xbox,platforms_Xbox 360,platforms_Xbox One,platforms_Xbox Series X|S,platforms_ZX Spectrum,platforms_Zeebo,platforms_e-Reader / Card-e Reader,platforms_iOS,platforms_visionOS
0,9953,92273,False,False,False,-1,2,False,-1,-1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1832,7153,True,True,True,2,0,False,0,0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,7987,57887,False,False,True,0,2,False,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,7,46076,False,False,False,-1,30,False,-1,-1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,10207,31256,False,False,False,-1,-1,False,-1,-1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
columns = ['offlinecoop', 'offlinecoopmax', 'offlinemax']

multiple_players = modes_df[columns[-1]] >= 1 

#Where multiple_players is true, replace the value of columns[-1] with 1
modes_df.loc[multiple_players, columns[-1]] = 1

#modes_df[columns[-1]].value_counts()
multiple_players

C:\Users\anees\AppData\Local\Temp\ipykernel_9420\4107046668.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  modes_df[multiple_players].loc[:, columns[-1]] = 1


0         True
1        False
2         True
3         True
4        False
         ...  
24023    False
24024    False
24025     True
24026     True
24027    False
Name: offlinemax, Length: 24028, dtype: bool

In [ ]:
full_df.groupby(columns)['name'].agg(lambda x: x.sample(1))